# Construção e manipulação manual do dataset da NBA

    - Este notebook foca na extração e tratamento de dados históricos da NBA (1996-2024) utilizando a biblioteca nba_api. O objetivo é construir um dataset robusto para modelos de Machine Learning, correlacionando estatísticas avançadas da temporada regular com o desempenho (vitórias) nos playoffs.

Realizamos a coleta iterativa de dados para cada temporada. Utilizamos o endpoint LeagueDashTeamStats com o parâmetro measure_type_detailed_defense='Advanced'. Isso nos permite obter métricas fundamentais como Offensive e Defensive Rating.

Nota técnica: Implementamos um time.sleep(1.5) para respeitar o rate limit da API da NBA e evitar bloqueios de requisição.

In [ ]:
import pandas as pd
import time
from nba_api.stats.endpoints import leaguedashteamstats

# definindo o intervalo de anos (ex: 1996 a 2024)
# o formato da NBA é "YYYY-YY" (ex: 1995-96)
anos = [f"{y}-{str(y+1)[-2:]}" for y in range(1996, 2024)]

dataset_completo = []

print("Iniciando coleta...")

for season in anos:
    try:
        # buscando estatísticas avançadas da temporada regular
        # o 'MeasureType' Advanced traz estatisticas avançadas que são cruciais
        data = leaguedashteamstats.LeagueDashTeamStats(
            season=season, 
            measure_type_detailed_defense='Advanced'
        )
        
        df_season = data.get_data_frames()[0]
        df_season['SEASON_ID'] = season # Adiciona a coluna do ano pra não misturar
        
        dataset_completo.append(df_season)
        print(f"Temporada {season} coletada com sucesso.")
        
        # pausa de 1.5s para não tomar ban por excesso de requisições
        time.sleep(1.5) 
        
    except Exception as e:
        print(f"Erro na temporada {season}: {e}")

# consolidando tudo em um único DataFrame
df_final = pd.concat(dataset_completo, ignore_index=True)

# salvando o progresso em CSV
df_final.to_csv("../CSVs/nba_team_stats_1980_2024.csv", index=False)
print("Arquivo salvo!")

Iniciando coleta...
Temporada 1996-97 coletada com sucesso.
Temporada 1997-98 coletada com sucesso.
Temporada 1998-99 coletada com sucesso.
Temporada 1999-00 coletada com sucesso.
Temporada 2000-01 coletada com sucesso.
Temporada 2001-02 coletada com sucesso.
Temporada 2002-03 coletada com sucesso.
Temporada 2003-04 coletada com sucesso.
Temporada 2004-05 coletada com sucesso.
Temporada 2005-06 coletada com sucesso.
Temporada 2006-07 coletada com sucesso.
Temporada 2007-08 coletada com sucesso.
Temporada 2008-09 coletada com sucesso.
Temporada 2009-10 coletada com sucesso.
Temporada 2010-11 coletada com sucesso.
Temporada 2011-12 coletada com sucesso.
Temporada 2012-13 coletada com sucesso.
Temporada 2013-14 coletada com sucesso.
Temporada 2014-15 coletada com sucesso.
Temporada 2015-16 coletada com sucesso.
Temporada 2016-17 coletada com sucesso.
Temporada 2017-18 coletada com sucesso.
Temporada 2018-19 coletada com sucesso.
Temporada 2019-20 coletada com sucesso.
Temporada 2020-21 co

Verificação rápida da estrutura dos dados brutos coletados para validar se as colunas e os IDs das temporadas foram inseridos corretamente antes de prosseguir para a engenharia de atributos.

In [ ]:
df_final.head()

,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,E_OFF_RATING,OFF_RATING,E_DEF_RATING,...,AST_RATIO_RANK,OREB_PCT_RANK,DREB_PCT_RANK,REB_PCT_RANK,TM_TOV_PCT_RANK,EFG_PCT_RANK,TS_PCT_RANK,PACE_RANK,PIE_RANK,SEASON_ID
0,1610612737,Atlanta Hawks,82,56,26,0.683,3961.0,105.4,106.4,98.9,...,28,14,9,8,13,12,12,26,4,1996-97
1,1610612738,Boston Celtics,82,15,67,0.183,3981.0,100.8,102.9,108.8,...,20,17,19,27,11,24,24,2,28,1996-97
2,1610612766,Charlotte Hornets,82,54,28,0.659,3961.0,108.0,109.1,105.6,...,3,27,16,20,5,3,2,22,10,1996-97
3,1610612741,Chicago Bulls,82,69,13,0.841,3946.0,111.1,112.4,99.2,...,2,2,10,2,2,5,7,17,2,1996-97
4,1610612739,Cleveland Cavaliers,82,42,40,0.512,3971.0,102.0,102.7,99.4,...,10,25,5,16,14,15,18,29,13,1996-97


Para que o modelo aprenda o que define um time vencedor, precisamos extrair o nosso target (variável dependente). Aqui, coletamos o LeagueGameLog filtrando apenas por jogos de 'Playoffs'. O objetivo é contabilizar o número total de vitórias de cada franquia em cada ano de pós-temporada.

In [ ]:
from nba_api.stats.endpoints import leaguegamelog

playoff_wins_list = []

for season in anos: # Usando a lista 'anos' feita anteriormente
    try:
        # coletando todos os jogos dos playoffs daquela temporada
        gamelog = leaguegamelog.LeagueGameLog(
            season=season,
            season_type_all_star='Playoffs'
        )
        df_playoffs = gamelog.get_data_frames()[0]
        
        # filtrando apenas as vitórias, e contando por time
        wins = df_playoffs[df_playoffs['WL'] == 'W'].groupby('TEAM_ID').size().reset_index(name='PLAYOFF_WINS')
        wins['SEASON_ID'] = season
        
        playoff_wins_list.append(wins)
        print(f"Vitórias dos Playoffs de {season} coletadas.")
        time.sleep(1.5) 
        
    except Exception as e:
        print(f"Erro nos playoffs de {season}: {e}")

df_playoff_target = pd.concat(playoff_wins_list, ignore_index=True)

Vitórias dos Playoffs de 1996-97 coletadas.
Vitórias dos Playoffs de 1997-98 coletadas.
Vitórias dos Playoffs de 1998-99 coletadas.
Vitórias dos Playoffs de 1999-00 coletadas.
Vitórias dos Playoffs de 2000-01 coletadas.
Vitórias dos Playoffs de 2001-02 coletadas.
Vitórias dos Playoffs de 2002-03 coletadas.
Vitórias dos Playoffs de 2003-04 coletadas.
Vitórias dos Playoffs de 2004-05 coletadas.
Vitórias dos Playoffs de 2005-06 coletadas.
Vitórias dos Playoffs de 2006-07 coletadas.
Vitórias dos Playoffs de 2007-08 coletadas.
Vitórias dos Playoffs de 2008-09 coletadas.
Vitórias dos Playoffs de 2009-10 coletadas.
Vitórias dos Playoffs de 2010-11 coletadas.
Vitórias dos Playoffs de 2011-12 coletadas.
Vitórias dos Playoffs de 2012-13 coletadas.
Vitórias dos Playoffs de 2013-14 coletadas.
Vitórias dos Playoffs de 2014-15 coletadas.
Vitórias dos Playoffs de 2015-16 coletadas.
Vitórias dos Playoffs de 2016-17 coletadas.
Vitórias dos Playoffs de 2017-18 coletadas.
Vitórias dos Playoffs de 2018-19

Exportação dos dados de vitórias coletados para um arquivo CSV, garantindo um checkpoint do progresso e evitando a necessidade de novas chamadas de API.

In [ ]:
df_playoff_target.to_csv("../CSVs/nba_team_playoffwins_1980_2024.csv", index=False)
print("Arquivo salvo!")

Arquivo salvo!


In [ ]:
df_playoff_target.head()

,TEAM_ID,PLAYOFF_WINS,SEASON_ID
0,1610612737,4,1996-97
1,1610612741,15,1996-97
2,1610612745,9,1996-97
3,1610612747,4,1996-97
4,1610612748,8,1996-97


Listagem de todas as colunas disponíveis no dataset consolidado para identificar redundâncias, variáveis categóricas desnecessárias e definir a estratégia de filtragem.

In [ ]:
df_final.columns

Index(['TEAM_ID', 'TEAM_NAME', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'E_OFF_RATING',
       'OFF_RATING', 'E_DEF_RATING', 'DEF_RATING', 'E_NET_RATING',
       'NET_RATING', 'AST_PCT', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT',
       'REB_PCT', 'TM_TOV_PCT', 'EFG_PCT', 'TS_PCT', 'E_PACE', 'PACE',
       'PACE_PER40', 'POSS', 'PIE', 'GP_RANK', 'W_RANK', 'L_RANK',
       'W_PCT_RANK', 'MIN_RANK', 'OFF_RATING_RANK', 'DEF_RATING_RANK',
       'NET_RATING_RANK', 'AST_PCT_RANK', 'AST_TO_RANK', 'AST_RATIO_RANK',
       'OREB_PCT_RANK', 'DREB_PCT_RANK', 'REB_PCT_RANK', 'TM_TOV_PCT_RANK',
       'EFG_PCT_RANK', 'TS_PCT_RANK', 'PACE_RANK', 'PIE_RANK', 'SEASON_ID'],
      dtype='str')

Removemos ruídos do dataset. Eliminamos colunas de 'Rank' (que são derivadas), nomes de times (para evitar viés de marca no modelo) e estatísticas básicas que já estão contempladas nas métricas avançadas. O foco é manter apenas as features que possuem poder preditivo real.

In [ ]:
# definindo aqui as colunas que quero ELIMINAR
cols_to_drop = [
    'TEAM_NAME',   # geralmente tiramos nomes para o modelo não "viciar" em times específicos
    'GP',
    'W', 
    'L',
    'MIN',
    'E_OFF_RATING',
    'E_DEF_RATING',
    'E_NET_RATING',
    'E_PACE',
    'GP_RANK', 
    'W_RANK', 
    'L_RANK',
    'W_PCT_RANK', 
    'MIN_RANK', 
    'OFF_RATING_RANK', 
    'DEF_RATING_RANK',
    'NET_RATING_RANK', 
    'AST_PCT_RANK', 
    'AST_TO_RANK', 
    'AST_RATIO_RANK',
    'OREB_PCT_RANK', 
    'DREB_PCT_RANK', 
    'REB_PCT_RANK', 
    'TM_TOV_PCT_RANK',
    'EFG_PCT_RANK', 
    'TS_PCT_RANK', 
    'PACE_RANK', 
    'PIE_RANK',
    'REB_PCT',
    'AST_PCT',
    'PACE_PER40', 
    'POSS',
    'EFG_PCT',
    'PIE'
]

# criando a cópia filtrada
# usando errors='ignore' para evitar erros caso seja digitado algum nome errado ou rode a célula duas vezes
df_stats_filtrado = df_final.drop(columns=cols_to_drop, errors='ignore').copy()

# visualizando o resultado
print(f"Colunas restantes: {df_stats_filtrado.columns.tolist()}")
display(df_stats_filtrado.head())

Colunas restantes: ['TEAM_ID', 'W_PCT', 'OFF_RATING', 'DEF_RATING', 'NET_RATING', 'AST_TO', 'AST_RATIO', 'OREB_PCT', 'DREB_PCT', 'TM_TOV_PCT', 'TS_PCT', 'PACE', 'SEASON_ID']


,TEAM_ID,W_PCT,OFF_RATING,DEF_RATING,NET_RATING,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,TM_TOV_PCT,TS_PCT,PACE,SEASON_ID
0,1610612737,0.683,106.4,100.3,6.2,1.27,15.5,0.334,0.670,0.168,0.542,88.54,1996-97
1,1610612738,0.183,102.9,110.1,-7.2,1.34,16.1,0.329,0.658,0.167,0.520,96.78,1996-97
2,1610612766,0.659,109.1,107.1,2.1,1.68,19.2,0.316,0.661,0.162,0.562,90.04,1996-97
3,1610612741,0.841,112.4,100.7,11.8,1.93,19.4,0.374,0.670,0.147,0.547,91.50,1996-97
4,1610612739,0.512,102.7,100.8,1.9,1.44,17.5,0.322,0.681,0.170,0.531,84.31,1996-97


In [ ]:
df_stats_filtrado.to_csv("../CSVs/nba_team_statsfiltrado_1980_2024.csv", index=False)
print("Arquivo salvo!")

Arquivo salvo!


Realizamos um left join entre as estatísticas da temporada regular e as vitórias nos playoffs.

Tratamento de Dados: Times que não se classificaram para os playoffs ou foram eliminados sem vencer jogos (varridas) resultarão em valores nulos na coluna PLAYOFF_WINS. Tratamos esses casos convertendo-os para 0, transformando o problema em uma escala quantitativa de sucesso.

In [ ]:
# fazendo o merge
print("\nRealizando o merge...")

# dataset com as 13 colunas restantes (11 features + 2 IDs)
df_ml = pd.merge(
    df_stats_filtrado, 
    df_playoff_target, 
    on=['TEAM_ID', 'SEASON_ID'], 
    how='left' # 'left' garante que todos os times da temp. regular continuem no dataset
)

# TRATAMENTO DOS ZEROS
# times que não foram pros playoffs, ou que foram e foram varridos (0-4), vão estar como NaN.
# nós preenchemos esses casos com 0 e convertemos para inteiro.
df_ml['PLAYOFF_WINS'] = df_ml['PLAYOFF_WINS'].fillna(0).astype(int)

print("\nMerge concluído com sucesso!")
print("\nAmostra dos dados finais:")
# mostrando algumas linhas para confirmar
display(df_ml.head(10))



Realizando o merge...

Merge concluído com sucesso!

Amostra dos dados finais:


,TEAM_ID,W_PCT,OFF_RATING,DEF_RATING,NET_RATING,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,TM_TOV_PCT,TS_PCT,PACE,SEASON_ID,PLAYOFF_WINS
0,1610612737,0.683,106.4,100.3,6.2,1.27,15.5,0.334,0.670,0.168,0.542,88.54,1996-97,4
1,1610612738,0.183,102.9,110.1,-7.2,1.34,16.1,0.329,0.658,0.167,0.520,96.78,1996-97,0
2,1610612766,0.659,109.1,107.1,2.1,1.68,19.2,0.316,0.661,0.162,0.562,90.04,1996-97,0
3,1610612741,0.841,112.4,100.7,11.8,1.93,19.4,0.374,0.670,0.147,0.547,91.50,1996-97,15
4,1610612739,0.512,102.7,100.8,1.9,1.44,17.5,0.322,0.681,0.170,0.531,84.31,1996-97,0
5,1610612742,0.293,99.5,106.0,-6.5,1.26,16.0,0.332,0.650,0.177,0.509,90.65,1996-97,0
6,1610612743,0.256,103.1,109.7,-6.6,1.39,17.4,0.324,0.666,0.175,0.530,93.67,1996-97,0
7,1610612765,0.659,108.5,102.6,5.9,1.49,16.1,0.302,0.675,0.146,0.554,86.09,1996-97,2
8,1610612744,0.366,105.3,110.3,-5.0,1.29,16.8,0.347,0.648,0.182,0.543,93.58,1996-97,0
9,1610612745,0.695,107.5,102.8,4.7,1.47,18.7,0.318,0.692,0.178,0.560,92.73,1996-97,9


In [ ]:
df_ml.to_csv("../CSVs/nba_ml_dataset_ready.csv", index=False)
print("\nDataset pronto e salvo!")


Dataset pronto e salvo!


Breve inspeção das medidas de tendência central e dispersão (média, desvio padrão, quartis) do dataset final pronto para o modelo.

In [ ]:
df_ml.describe()

,TEAM_ID,W_PCT,OFF_RATING,DEF_RATING,NET_RATING,AST_TO,AST_RATIO,OREB_PCT,DREB_PCT,TM_TOV_PCT,TS_PCT,PACE,PLAYOFF_WINS
count,8.320000e+02,832.000000,832.000000,832.000000,832.000000,832.000000,832.000000,832.000000,832.000000,832.00000,832.000000,832.000000,832.000000
mean,1.610613e+09,0.499817,106.168750,106.161418,0.006250,1.558486,17.026322,0.298554,0.701139,0.15243,0.541391,94.723377,2.729567
std,8.617705e+00,0.152469,4.842545,4.682845,4.786967,0.227024,1.290982,0.030061,0.025516,0.01439,0.025661,3.860714,4.257386
min,1.610613e+09,0.106000,91.200000,93.100000,-15.000000,1.010000,13.000000,0.216000,0.602000,0.11200,0.468000,84.310000,0.000000
25%,1.610613e+09,0.390000,102.875000,102.800000,-3.225000,1.400000,16.100000,0.277000,0.683000,0.14200,0.523000,91.680000,0.000000
50%,1.610613e+09,0.512000,105.500000,105.900000,0.300000,1.540000,17.000000,0.299000,0.701000,0.15200,0.539000,94.140000,0.000000
75%,1.610613e+09,0.610000,109.600000,109.400000,3.400000,1.700000,17.800000,0.321000,0.719000,0.16200,0.559000,97.632500,4.000000
max,1.610613e+09,0.890000,122.200000,119.600000,11.800000,2.380000,21.200000,0.383000,0.775000,0.20800,0.610000,105.510000,16.000000


Em vez de um split aleatório, optamos por um Split Temporal. Utilizamos todo o histórico disponível para treino e reservamos a temporada mais recente (2023-24) para teste. Essa abordagem simula de forma realística a capacidade do modelo de prever o futuro com base no passado.

In [ ]:
# identificando o último ano dinamicamente (pra se o dataset for expandido posteriormente)
ultimo_ano = df_ml['SEASON_ID'].max()

# definindo Identificadores e Alvo
identificadores = ['TEAM_ID', 'SEASON_ID']
alvo = 'PLAYOFF_WINS'

# Split: Tudo antes do último ano é treino, o último ano é teste
df_train_raw = df_ml[df_ml['SEASON_ID'] < ultimo_ano].copy()
df_test_raw = df_ml[df_ml['SEASON_ID'] == ultimo_ano].copy()

# separando X e y (Removendo IDs do treinamento)
X_train_raw = df_train_raw.drop(columns=identificadores + [alvo])
y_train = df_train_raw[alvo]

X_test_raw = df_test_raw.drop(columns=identificadores + [alvo])
y_test = df_test_raw[alvo]

print(f"Temporada de Teste: {ultimo_ano}")
print(f"Treino: {X_train_raw.shape[0]} amostras | Teste: {X_test_raw.shape[0]} amostras")

Temporada de Teste: 2023-24
Treino: 802 amostras | Teste: 30 amostras


Como as métricas possuem ordens de magnitude diferentes (ex: PACE ~100 vs W_PCT ~0.5), aplicamos o StandardScaler.

Atenção: O ajuste (fit) é feito exclusivamente nos dados de treino para evitar o data leakage (vazamento de informação dos dados de teste para o treino).

In [ ]:
from sklearn.preprocessing import StandardScaler

# inicializando e ajustando apenas nos dados de treino
scaler = StandardScaler()
scaler.fit(X_train_raw)

# aplicando a transformação em ambos
X_train_norm = scaler.transform(X_train_raw)
X_test_norm = scaler.transform(X_test_raw)

# convertendo de volta para DataFrame para manter os nomes das colunas
X_train = pd.DataFrame(X_train_norm, columns=X_train_raw.columns)
X_test = pd.DataFrame(X_test_norm, columns=X_test_raw.columns)

print("Normalização concluída com base no histórico pré-teste.")

Normalização concluída com base no histórico pré-teste.


Fase final de persistência. Salvamos os conjuntos de treino e teste já normalizados, além do objeto scaler. Isso permite que o treinamento do modelo e as futuras inferências utilizem exatamente a mesma escala estatística.

In [ ]:
import joblib # Útil para salvar o objeto scaler

# salvando os DataFrames (incluindo IDs para não se perder)
# concatenamos os IDs de volta apenas para o CSV ficar legível
X_train_to_save = pd.concat([df_train_raw[identificadores].reset_index(drop=True), X_train], axis=1)
X_test_to_save = pd.concat([df_test_raw[identificadores].reset_index(drop=True), X_test], axis=1)

X_train_to_save.to_csv('../CSVs/X_train_final.csv', index=False)
y_train.to_frame().to_csv('../CSVs/y_train_final.csv', index=False)
X_test_to_save.to_csv('../CSVs/X_test_final.csv', index=False)
y_test.to_frame().to_csv('../CSVs/y_test_final.csv', index=False)

# salvando o Scaler (Opcional, mas muito recomendado)
joblib.dump(scaler, 'nba_scaler.pkl')

print("Arquivos de treino, teste e scaler salvos com sucesso!")

Arquivos de treino, teste e scaler salvos com sucesso!
